In [ ]:
import pandas as pd
import numpy as np


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie_raw_object_id.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia

In [ ]:
data_gdynia_long_categories = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long_categories.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long_categories.head()

1.

In [ ]:
bud = data_gdynia.groupby(["addr_street", "arch_style"]).size()
bud.head()

In [ ]:
bud.loc["Świętojańska"]

In [ ]:
bud.xs("klasycyzm", level="arch_style")

In [ ]:
assert bud.index.nlevels == 2
assert bud.sum() == data_gdynia["addr_street"].notna().sum()

2.

In [ ]:
szer = bud.unstack()
szer0 = bud.unstack(fill_value=0)
szer.dtypes

In [ ]:
assert szer.isna().sum().sum() == 126
assert (szer0.sum(axis=1) == data_gdynia.groupby("addr_street").size()).all()

3.

In [ ]:
pd.testing.assert_series_equal(szer0.stack().loc[szer0.stack() > 0], bud)

In [ ]:
szer.stack()

unstack stack witgout filling Nan values, leaves Nan values in the series

4.

In [ ]:
arch_style_year = data_gdynia.groupby(["arch_style", "build_year_est"]).size()
arch_style_year

In [ ]:
arch_style_year.unstack()

In [ ]:
arch_style_year.unstack("arch_style")

In [ ]:
arch_style_year.unstack(level=0)

In [ ]:
pd.testing.assert_frame_equal(
    arch_style_year.unstack().T, arch_style_year.unstack("arch_style")
)
pd.testing.assert_frame_equal(
    arch_style_year.unstack(level=0), arch_style_year.unstack("arch_style")
)

5.

In [ ]:
t = bud.reset_index(name="num_build")
t.pivot(index="addr_street", columns="arch_style", values="num_build")

In [ ]:
try:
    data_gdynia.pivot(
        index="addr_street", columns="arch_style", values="number_of_storeys"
    )
except ValueError as e:
    print(e)

In [ ]:
pd.testing.assert_frame_equal(t.pivot(index="addr_street", columns="arch_style", values="num_build"), szer)

6.

In [ ]:
def only_big_streets(df, min_build):
    return df.loc[df.groupby("addr_street").transform("size") >= min_build]

big = data_gdynia.pipe(only_big_streets, min_build = 20)


In [ ]:
pd.testing.assert_frame_equal(big.groupby(['addr_street','arch_style'])['number_of_storeys'].median().unstack(), big.pivot_table(index = 'addr_street', columns = 'arch_style', values = 'number_of_storeys', aggfunc = 'median'))

In [ ]:
big.pivot_table(index = 'addr_street', columns = 'arch_style', values = 'number_of_storeys')

pivot table counts mean by default

7.

In [ ]:
data_gdynia.pivot_table(index = 'arch_style', columns = 'condition', values = 'osm_id', aggfunc = 'count', fill_value= 0, margins = True, margins_name= 'Sum')


8.

In [ ]:
tab = data_gdynia.pivot_table(index = 'arch_style', columns = 'condition', values = 'osm_id', aggfunc = 'count', fill_value= 0, margins = False)
tab

In [ ]:
style_condition = tab.div(tab.sum(axis =1 ), axis = 0)
style_condition

In [ ]:
assert np.isclose(style_condition.sum(axis = 1), 1).all()

In [ ]:
szer0

In [ ]:
szer0.reset_index()


9.

In [ ]:
m = szer0.reset_index().melt(id_vars = 'addr_street', var_name = 'arch_style', value_name = 'num_build') 
m

In [ ]:
pd.testing.assert_frame_equal(t,m.loc[m['num_build'] != 0].sort_values(['addr_street', 'arch_style'], ascending= True).reset_index(drop= True) )

10.

In [ ]:
lok = data_gdynia_long_categories.join(data_gdynia[['arch_style']], validate= 'many_to_one')
p10 = lok.pivot_table(index= 'category', columns = 'arch_style', aggfunc='size', fill_value=0)
p10 = p10.reset_index().melt(id_vars = 'category', var_name = 'arch_style', value_name = 'num_build')
p10 = p10.loc[p10['num_build']>0].set_index(['category', 'arch_style']).sort_index()

In [ ]:
pd.testing.assert_series_equal(p10['num_build'], lok.groupby(['category','arch_style']).size().rename('num_build'))

In [ ]:
data_gdynia.groupby(['addr_street','arch_style']).size().reset_index(name = 'size').pivot(index= 'addr_street', columns = 'arch_style', values = 'size').reset_index().melt(id_vars= 'addr_street', var_name= 'archi_style', value_name= 'build_num')